### Importing The Required Libraries 

In [1]:
from typing import TypedDict,Annotated 
from langgraph.graph import StateGraph,START,END
import sqlite3
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph.message import add_messages
from langchain.messages import HumanMessage,SystemMessage

C:\Users\rajve\miniconda3\envs\nlp_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Defining The Agent State

In [2]:
class AgentState(TypedDict):
    messages:Annotated[list,add_messages]

### Defining The llm 

In [3]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

### Defining The Chatbot Node

In [13]:
def chatbot(state:AgentState):
    print("Reading Memory And generating Responses")
    sys_msg=SystemMessage(content="You are a helpful software customer support assistant.")
    message=[sys_msg]+state["messages"]
    response=llm.invoke(message)
    return {"messages":[response]}

### Making connection With The Database

In [14]:
conn=sqlite3.connect("agent.db",check_same_thread=False)

In [15]:
memory=SqliteSaver(conn)

### Making The Graph Builder

In [16]:
builder=StateGraph(AgentState)

In [17]:
builder.add_node("agent",chatbot)

In [18]:
builder.add_edge(START,"agent")
builder.add_edge("agent",END)

### Compiling The Graph

In [34]:
config={
    "configurable":{
        "thread_id":"1"
    }
}

In [36]:
graph=builder.compile(checkpointer=memory)

In [37]:
inp={"messages":[HumanMessage(content="Hi, my name is Alex and my dashboard keeps crashing.")]}


In [38]:
state_1=graph.invoke(inp,config=config)

Reading Memory And generating Responses


In [39]:
state_1

{'messages': [HumanMessage(content='Hi, my name is Alex and my dashboard keeps crashing.', additional_kwargs={}, response_metadata={}, id='abecb4bc-a0c1-4aad-b4cb-8fde14ac0e94'),
  AIMessage(content="Hi Alex, I'm sorry to hear your dashboard keeps crashing – that sounds really frustrating!\n\nTo help me understand what might be going on, could you please tell me a little more about it?\n\n1.  **Which dashboard are you referring to?** (e.g., a specific product name, a custom dashboard, etc.)\n2.  **When did this start happening?** Was it working fine before?\n3.  **Are you seeing any specific error messages** when it crashes? If so, what do they say?\n4.  **What browser are you using** (Chrome, Firefox, Edge, Safari, etc.) and on what device (desktop, laptop, tablet)?\n5.  **Have you tried any troubleshooting steps already?** (e.g., refreshing the page, restarting your computer, trying a different browser)\n\nThe more details you can provide, the quicker I can help you get this resolved

In [40]:
inp_2={"messages":"What is my name"}

In [41]:
state_2=graph.invoke(inp_2,config=config)

Reading Memory And generating Responses


In [42]:
state_2

{'messages': [HumanMessage(content='Hi, my name is Alex and my dashboard keeps crashing.', additional_kwargs={}, response_metadata={}, id='abecb4bc-a0c1-4aad-b4cb-8fde14ac0e94'),
  AIMessage(content="Hi Alex, I'm sorry to hear your dashboard keeps crashing – that sounds really frustrating!\n\nTo help me understand what might be going on, could you please tell me a little more about it?\n\n1.  **Which dashboard are you referring to?** (e.g., a specific product name, a custom dashboard, etc.)\n2.  **When did this start happening?** Was it working fine before?\n3.  **Are you seeing any specific error messages** when it crashes? If so, what do they say?\n4.  **What browser are you using** (Chrome, Firefox, Edge, Safari, etc.) and on what device (desktop, laptop, tablet)?\n5.  **Have you tried any troubleshooting steps already?** (e.g., refreshing the page, restarting your computer, trying a different browser)\n\nThe more details you can provide, the quicker I can help you get this resolved

In [45]:
conn.close()